# **CELL 1 — Install Dependencies**

In [ ]:
!pip install transformers datasets scikit-learn pandas torch

# **CELL 2 — Import Libraries**

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

import torch

# **CELL 3 — Load Dataset and data cleaning**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = '/content/drive/My Drive/Datasets/Phishing_Email.csv'
df = pd.read_csv(file_path)
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df = df.dropna()

# **CELL 4 — Train/Test Split**

In [ ]:
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['Email Text'],
    df['Email Type'],
    test_size=0.30,
    random_state=42,
    stratify=df['Email Type']  # IMPORTANT (keeps class balance)
)

In [ ]:
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.50,   # 50% of 30% = 15%
    random_state=42,
    stratify=temp_labels
)

# ***BASELINE MODEL (LOGISTIC REGRESSION)***

# **CELL 5 — TF-IDF + Logistic Regression**

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, train_labels)

lr_preds = lr_model.predict(X_test_tfidf)

print("Logistic Regression Results:")
print(classification_report(test_labels, lr_preds))

Logistic Regression Results:
                precision    recall  f1-score   support

Phishing Email       0.94      0.96      0.95      1097
    Safe Email       0.97      0.96      0.97      1699

      accuracy                           0.96      2796
     macro avg       0.96      0.96      0.96      2796
  weighted avg       0.96      0.96      0.96      2796



# ***DISTILBERT MODEL***



# **CELL 6 — Tokenization and **

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
#This converts your raw email text → numbers that DistilBERT can understand
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=256)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# **CELL 7 — Dataset Class**

In [ ]:
class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels # Labels are already mapped to integers

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Map string labels to integers
label_map = {'Safe Email': 0, 'Phishing Email': 1}
train_labels_encoded = train_labels.map(label_map).tolist()
test_labels_encoded = test_labels.map(label_map).tolist()
val_labels_encoded = val_labels.map(label_map).tolist()

train_dataset = EmailDataset(train_encodings, train_labels_encoded)
test_dataset = EmailDataset(test_encodings, test_labels_encoded)
eval_dataset = EmailDataset(val_encodings, val_labels_encoded) # Corrected to use val_labels_encoded

# **CELL 8 — Load Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# **CELL 9 — Training Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    logging_dir="./logs",
    save_strategy="epoch"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# **CELL 10 — Trainer**

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer, # renamed  processing_class for better compatibility
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.047002,0.121448,0.973524,0.966055,0.972299,0.959891
2,0.029461,0.074305,0.982826,0.978320,0.969561,0.987238


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3262, training_loss=0.04431497765639894, metrics={'train_runtime': 740.4997, 'train_samples_per_second': 35.228, 'train_steps_per_second': 4.405, 'total_flos': 1727772280670208.0, 'train_loss': 0.04431497765639894, 'epoch': 2.0})

# **CELL 11 — Evaluation**

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)

print("DistilBERT Results:")
print(classification_report(test_labels_encoded, preds))

DistilBERT Results:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      1699
           1       0.96      0.98      0.97      1097

    accuracy                           0.98      2796
   macro avg       0.98      0.98      0.98      2796
weighted avg       0.98      0.98      0.98      2796



# ***PREDICTION FUNCTION***

# **LLM EXPLANATION**:

In [ ]:
def explain_email(text, prediction):
    text_lower = text.lower()

    reasons = []

    # 🚨 Urgency indicators
    urgency_keywords = ["urgent", "immediately", "now", "asap", "suspended", "limited time"]
    if any(word in text_lower for word in urgency_keywords):
        reasons.append("The email creates urgency (e.g., 'urgent', 'immediately'), which is common in phishing attacks.")

    # 🔐 Credential / verification requests
    credential_keywords = ["verify", "login", "password", "account", "confirm", "update"]
    if any(word in text_lower for word in credential_keywords):
        reasons.append("The email asks for account verification or login details, which may indicate credential harvesting.")

    # 🔗 Suspicious links
    if "http" in text_lower or "www" in text_lower:
        reasons.append("The email contains links, which could redirect to malicious websites.")

    # 💰 Financial bait
    money_keywords = ["bank", "payment", "invoice", "transfer", "refund", "prize"]
    if any(word in text_lower for word in money_keywords):
        reasons.append("The email references financial matters, a common phishing tactic.")

    # ⚠️ Threat / fear language
    threat_keywords = ["suspended", "blocked", "terminated", "legal action"]
    if any(word in text_lower for word in threat_keywords):
        reasons.append("The email uses threatening language to pressure the user.")

    # 🧠 Final response
    if reasons:
        return " | ".join(reasons)
    else:
        return "No strong phishing indicators found."

# **CELL 12 — Predict Function**

In [ ]:
def predict_email(text):
    # Tokenize input
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)

    # Move inputs to the same device as the model
    inputs = {key: val.to(model.device) for key, val in inputs.items()}

    # Model prediction
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    confidence = torch.max(probs).item()
    prediction = torch.argmax(probs).item()

    label = "Phishing" if prediction == 1 else "Safe"

    # 🧠 Call explanation function
    explanation = explain_email(text, label)

    # 🎯 Final structured output
    return {
        "prediction": label,
        "confidence": round(confidence, 4),
        "explanation": explanation
    }

# **TEST IT**

In [ ]:
predict_email("Your account has been suspended. Click here to verify immediately!")


{'prediction': 'Phishing',
 'confidence': 0.9999,
 'explanation': "The email creates urgency (e.g., 'urgent', 'immediately'), which is common in phishing attacks. | The email asks for account verification or login details, which may indicate credential harvesting. | The email uses threatening language to pressure the user."}